# Carbon-Aware Recommender System — Full Pipeline

This notebook runs the complete pipeline on Google Colab (with GPU):

| Step | Script | Description |
|------|--------|-------------|
| 1 | `01_train_recommender.py` | RecBole → top-K candidates with `relevance_score` |
| 2 | `02_train_deepfm.py` | DeepFM → `engagement_score` per candidate |
| 3 | `03_rerank.py` | Carbon-aware re-ranking: `s = (1−λ)·engagement − λ·carbon` |
| 4 | `04_evaluate.py` | Pareto frontier, summary tables, plots |

**Runtime**: Select **GPU** under *Runtime → Change runtime type* for faster training.

## 0. Setup

In [2]:
# ── Clone the repository ──────────────────────────────────────────
import os

REPO_URL = "https://github.com/andersvestrum/carbon-aware-recsys.git"
BRANCH = "carbon_catalogue"
REPO_DIR = "/content/carbon-aware-recsys"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin {BRANCH}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

### ⚠️ Data Setup

The `data/` folder is gitignored (too large for GitHub), so you need to upload it.

**Option A — Google Drive (recommended):**
1. Upload `data/interim/` to your Google Drive (e.g. `My Drive/carbon-aware-recsys/data/interim/`)
2. Run the cell below to mount Drive and copy the data

**Option B — Direct upload:**
Skip the cell below and manually upload `data/interim/` via the Colab file browser (folder icon on the left).

In [3]:
# ── Mount Google Drive & copy data ────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# ── Configure your Drive path here ───────────────────────────────
# Point this to wherever you uploaded the project data on Drive.
DRIVE_DATA = "/content/drive/MyDrive/carbon-aware-recsys/data"

import shutil
from pathlib import Path

for folder in ["interim", "raw", "processed"]:
    src = Path(DRIVE_DATA) / folder
    dst = Path(REPO_DIR) / "data" / folder
    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(str(src), str(dst), dirs_exist_ok=True)  # Robust: allow existing dirs
        print(f"  ✓ Copied {folder}/ ({sum(1 for _ in dst.rglob('*') if _.is_file())} files)")
    else:
        print(f"  ⚠️ {src} not found on Drive — skipping")

In [10]:
# ── Install dependencies ──────────────────────────────────────────
!pip install -q torch recbole deepctr-torch scikit-learn \
    pandas numpy matplotlib seaborn pyyaml joblib tqdm pyarrow \
    omegaconf colorlog

In [5]:
# ── Verify GPU and imports ────────────────────────────────────────
import torch
import numpy as np
import pandas as pd

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} — device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected. Training will be slow. "
          "Go to Runtime → Change runtime type → GPU.")

In [6]:
# ── Add project root to sys.path ─────────────────────────────────
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Quick sanity check
from src.recommender.recbole_formatter import format_category_for_recbole
from src.engagement.deepfm import DeepFMWrapper
from src.reranking.carbon_reranker import CarbonReranker
from src.evaluation.metrics import pareto_frontier
print("All modules imported ✓")

In [7]:
# ── Verify data exists ───────────────────────────────────────────
from pathlib import Path

INTERIM_DIR = Path("data/interim")
for split in ["train", "val", "test"]:
    for cat in ["electronics", "home_and_kitchen", "sports_and_outdoors"]:
        p = INTERIM_DIR / split / f"{cat}.csv"
        if p.exists():
            n = sum(1 for _ in open(p)) - 1
            print(f"  ✓ {p}  ({n:,} rows)")
        else:
            print(f"  ✗ {p}  MISSING")

## Configuration

Edit these to control what gets trained:

In [8]:
# ── Pipeline configuration ───────────────────────────────────────
CATEGORIES = ["electronics", "home_and_kitchen", "sports_and_outdoors"]
MODELS = ["BPR", "NeuMF", "SASRec", "LightGCN"]  # List of models to loop over
TOP_K_CANDIDATES = 100        # candidates per user from RecBole
DEEPFM_EPOCHS = 3             # Fewer epochs for dev run
DEEPFM_BATCH_SIZE = 512       # larger batch → faster on GPU
TOP_K_RERANK = 10             # final list length per user

# λ values for carbon-aware re-ranking sweep
LAMBDA_VALUES = [0.0, 0.25, 0.5, 0.75, 0.80, 0.85, 0.9, 0.95,1.0]  # Fewer values for speed

print(f"Categories:  {CATEGORIES}")
print(f"Models:      {MODELS}")
print(f"Device:      {DEVICE}")
print(f"λ values:    {LAMBDA_VALUES}")

---
## Step 1 — RecBole: Candidate Generation

In [11]:
from src.recommender.recbole_formatter import format_category_for_recbole
from src.recommender.trainer import train_and_score

recbole_results = {}

N_USERS_DEBUG = 500  # Limit to first 500 users for speed
EPOCHS_DEBUG = 2     # Fewer epochs for dev run
BATCH_SIZE_DEBUG = 1024

for model in MODELS:
    for cat in CATEGORIES:
        print(f"\n{'='*60}")
        print(f"Step 1 — RecBole ({model}) on {cat}")
        print(f"{'='*60}")

        # Format interim data → RecBole .inter / .item files
        output_dir, dataset_name = format_category_for_recbole(cat)
        print(f"  RecBole dataset: {dataset_name}")

        # Check for model-specific config
        config_file = Path(f"configs/recbole/{model.lower()}.yaml")
        config_file = config_file if config_file.exists() else None

        # Fast dev overrides
        overrides = {
            "device": DEVICE,
            "epochs": EPOCHS_DEBUG,
            "train_batch_size": BATCH_SIZE_DEBUG,
            "eval_batch_size": BATCH_SIZE_DEBUG,
        } if DEVICE == "cuda" else {
            "epochs": EPOCHS_DEBUG,
            "train_batch_size": BATCH_SIZE_DEBUG,
            "eval_batch_size": BATCH_SIZE_DEBUG,
        }

        # Train and extract relevance scores
        scores_df, eval_results = train_and_score(
            dataset_name=dataset_name,
            model_name=model,
            config_file=config_file,
            overrides=overrides,
            top_k=TOP_K_CANDIDATES,
        )

        recbole_results[(cat, model)] = {"scores": scores_df, "eval": eval_results}

        print(f"\n  RecBole eval: {eval_results}")
        print(f"  Scores: {len(scores_df):,} (user, item) pairs")
        print(f"  Sample:")
        display(scores_df.head())

---
## Step 2 — DeepFM: Engagement Prediction

In [ ]:
# --- Step 2 — DeepFM: Engagement Prediction (SKIPPED) ---

# Instead, use RecBole scores as engagement scores for re-ranking
deepfm_results = {}

for cat in CATEGORIES:
    print(f"\n{'='*60}")
    print(f"Step 2 — SKIP DeepFM: Using RecBole relevance_score as engagement for {cat}")
    print(f"{'='*60}")

    # Use RecBole scores as engagement
    engagement_df = recbole_results[cat]["scores"].rename(columns={"relevance_score": "engagement_score"})
    history = {}  # Empty, since no DeepFM training

    deepfm_results[cat] = {"engagement": engagement_df, "history": history}

    print(f"\n  Engagement scores: {len(engagement_df):,} pairs")
    print(f"  Score range: [{engagement_df['engagement_score'].min():.4f}, "
          f"{engagement_df['engagement_score'].max():.4f}]")
    print(f"  Sample:")
    display(engagement_df.head())

In [ ]:
# ── Plot DeepFM training curves ──────────────────────────────────
import matplotlib.pyplot as plt

for cat, res in deepfm_results.items():
    history = res["history"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Loss
    if "binary_crossentropy" in history:
        axes[0].plot(history["binary_crossentropy"], label="Train")
        if "val_binary_crossentropy" in history:
            axes[0].plot(history["val_binary_crossentropy"], label="Val")
        axes[0].set_title(f"Binary Cross-Entropy — {cat}")
        axes[0].set_xlabel("Epoch")
        axes[0].legend()

    # AUC
    if "auc" in history:
        axes[1].plot(history["auc"], label="Train")
        if "val_auc" in history:
            axes[1].plot(history["val_auc"], label="Val")
        axes[1].set_title(f"AUC — {cat}")
        axes[1].set_xlabel("Epoch")
        axes[1].legend()

    fig.tight_layout()
    plt.show()

---
## Step 3 — Carbon-Aware Re-ranking

In [ ]:
from src.recommender.recbole_formatter import _concat_interim_splits
from src.reranking.carbon_reranker import (
    CarbonReranker,
    build_test_set,
    compute_reranking_metrics,
)

N_USERS_DEBUG = 500  # Limit to first 500 users for fast debugging

reranking_results = {}

for cat in CATEGORIES:
    print(f"\n{'='*60}")
    print(f"Step 3 — Carbon re-ranking on {cat}")
    print(f"{'='*60}")

    # Load engagement scores
    engagement_df = deepfm_results[cat]["engagement"]
    # Limit to N_USERS_DEBUG users
    user_subset = engagement_df["user_id"].unique()[:N_USERS_DEBUG]
    engagement_df = engagement_df[engagement_df["user_id"].isin(user_subset)].copy()

    # Load carbon data from interim
    interactions = _concat_interim_splits(cat)
    carbon_df = (
        interactions[["parent_asin", "pcf"]]
        .drop_duplicates(subset="parent_asin")
        .copy()
    )
    print(f"  Carbon data: {len(carbon_df):,} items, "
          f"median pcf = {carbon_df['pcf'].median():.2f} kg CO₂e")

    # Build test set and limit to N_USERS_DEBUG users
    test_interactions = build_test_set(interactions)
    test_interactions = test_interactions[test_interactions["user_id"].isin(user_subset)].copy()
    print(f"  Test set: {len(test_interactions):,} users (limited to {N_USERS_DEBUG})")

    # Sweep λ
    reranker = CarbonReranker(top_k=TOP_K_RERANK)
    all_metrics = []

    for lam in LAMBDA_VALUES:
        print(f"  Reranking for λ={lam:.3f} ...")
        ranked = reranker.rerank(engagement_df, carbon_df, lam)
        metrics = compute_reranking_metrics(ranked, test_interactions, lam, k=TOP_K_RERANK)
        all_metrics.append(metrics)

        # Save re-ranked lists
        out_path = Path(f"output/results/{cat}_{RECBOLE_MODEL}_reranked_{lam:.3f}.parquet")
        out_path.parent.mkdir(parents=True, exist_ok=True)
        ranked.to_parquet(out_path, index=False)

    reranking_results[cat] = all_metrics

    # Save metrics JSON (for step 4)
    import json
    baseline = next(m for m in all_metrics if m["lambda"] == 0.0)
    greenest = min(all_metrics, key=lambda m: m["avg_carbon_kg"])
    summary = {
        "category": cat,
        "model": RECBOLE_MODEL,
        "baseline_carbon_kg": baseline["avg_carbon_kg"],
        "baseline_NDCG@10": baseline["NDCG@10"],
        "greenest_lambda": greenest["lambda"],
        "greenest_carbon_kg": greenest["avg_carbon_kg"],
        "greenest_NDCG@10": greenest["NDCG@10"],
    }
    metrics_path = Path(f"output/results/{cat}_{RECBOLE_MODEL}_reranking_metrics.json")
    with open(metrics_path, "w") as f:
        json.dump({"summary": summary, "per_lambda": all_metrics}, f, indent=2)

    # Print results table
    print(f"\n  {'λ':>6}  {'NDCG@10':>8}  {'Recall@10':>10}  {'MRR':>6}  {'Avg CO₂ (kg)':>13}")
    print(f"  {'─'*6}  {'─'*8}  {'─'*10}  {'─'*6}  {'─'*13}")
    for m in all_metrics:
        print(f"  {m['lambda']:6.3f}  {m['NDCG@10']:8.4f}  {m['Recall@10']:10.4f}  "
              f"{m['MRR']:6.4f}  {m['avg_carbon_kg']:13.2f}")

---
## Step 4 — Evaluation & Pareto Frontier

In [ ]:
from src.evaluation.metrics import (
    pareto_frontier,
    build_summary_table,
    plot_tradeoff_curve,
    plot_lambda_sensitivity,
    plot_multi_category,
)

for cat in CATEGORIES:
    print(f"\n{'='*60}")
    print(f"Step 4 — Evaluation: {cat}")
    print(f"{'='*60}")

    metrics = reranking_results[cat]

    # ── Pareto frontier ───────────────────────────────────────────
    front = pareto_frontier(metrics)
    print(f"\n  Pareto-optimal points ({len(front)}):")
    for pt in front:
        baseline_carbon = metrics[0]["avg_carbon_kg"]
        reduction = 100 * (baseline_carbon - pt["avg_carbon_kg"]) / max(baseline_carbon, 1e-9)
        print(f"    λ={pt['lambda']:.3f}  NDCG@10={pt['NDCG@10']:.4f}  "
              f"carbon={pt['avg_carbon_kg']:.2f} kg  (−{reduction:.1f}%)")

    # ── Summary table ─────────────────────────────────────────────
    summary_df = build_summary_table(metrics)
    summary_df.to_csv(f"output/results/{cat}_{RECBOLE_MODEL}_evaluation_summary.csv", index=False)
    display(summary_df)

    # ── Best trade-off point ──────────────────────────────────────
    baseline_carbon = metrics[0]["avg_carbon_kg"]
    best = None
    for pt in sorted(front, key=lambda p: -p["NDCG@10"]):
        reduction = (baseline_carbon - pt["avg_carbon_kg"]) / max(baseline_carbon, 1e-9)
        if reduction >= 0.10:
            best = pt
            break
    if best:
        print(f"\n  ★ Recommended operating point: λ={best['lambda']:.3f}")
        print(f"    NDCG@10={best['NDCG@10']:.4f}  carbon={best['avg_carbon_kg']:.2f} kg")

In [ ]:
# ── Trade-off plot (per category) ────────────────────────────────
for cat in CATEGORIES:
    fig = plot_tradeoff_curve(
        reranking_results[cat], cat, RECBOLE_MODEL,
        save=True, show=True,
    )

In [ ]:
# ── λ sensitivity plot (per category) ────────────────────────────
for cat in CATEGORIES:
    fig = plot_lambda_sensitivity(
        reranking_results[cat], cat, RECBOLE_MODEL,
        save=True, show=True,
    )

In [ ]:
# ── Multi-category comparison (if > 1 category) ──────────────────
if len(CATEGORIES) > 1:
    fig = plot_multi_category(
        reranking_results, RECBOLE_MODEL,
        save=True, show=True,
    )
else:
    print("Only one category — skipping multi-category plot. "
          "Add more categories to CATEGORIES above to enable.")

In [ ]:
# --- Multi-Model Pareto Plot: BPR, NeuMF, SASRec, LightGCN ---
from src.evaluation.metrics import pareto_frontier
import matplotlib.pyplot as plt
import json

models = ["BPR", "NeuMF", "SASRec", "LightGCN"]
category = CATEGORIES[0]  # Plot for the first category in CATEGORIES
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

plt.figure(figsize=(8, 6))

for model, color in zip(models, colors):
    metrics_path = f"output/results/{category}_{model}_reranking_metrics.json"
    try:
        with open(metrics_path, "r") as f:
            metrics_data = json.load(f)["per_lambda"]
        front = pareto_frontier(metrics_data)
        x = [pt["avg_carbon_kg"] for pt in front]
        y = [pt["NDCG@10"] for pt in front]
        plt.plot(x, y, marker="o", label=model, color=color)
    except Exception as e:
        print(f"Could not load results for {model}: {e}")

plt.xlabel("Average Carbon (kg CO₂e)")
plt.ylabel("NDCG@10")
plt.title(f"Pareto Frontier Comparison — {category}")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

---
## 📥 Download Results

Run the cell below to zip all outputs and download them.

In [ ]:
# ── Zip and download results ─────────────────────────────────────
import shutil

shutil.make_archive("carbon_aware_results", "zip", ".", "output")
print("Created carbon_aware_results.zip")

# Uncomment to auto-download in Colab:
# from google.colab import files
# files.download("carbon_aware_results.zip")

---
## Summary

| Output | Location |
|--------|----------|
| RecBole scores | `output/results/<cat>_<model>_scores.parquet` |
| Engagement scores | `output/results/<cat>_<model>_engagement.parquet` |
| Re-ranked lists | `output/results/<cat>_<model>_reranked_<λ>.parquet` |
| Metrics JSON | `output/results/<cat>_<model>_reranking_metrics.json` |
| Summary CSV | `output/results/<cat>_<model>_evaluation_summary.csv` |
| Pareto JSON | `output/results/<cat>_<model>_pareto.json` |
| Trade-off plot | `output/figures/<cat>_<model>_tradeoff.png` |
| λ-sensitivity plot | `output/figures/<cat>_<model>_lambda_sensitivity.png` |
| DeepFM model | `output/models/deepfm_<cat>_<model>.joblib` |

# --- Summary of Best Results and Plots ---
import json
import matplotlib.pyplot as plt

print("=== Best Pareto Points (≥10% carbon reduction) ===")
best_points = []
for cat in CATEGORIES:
    for model in MODELS:
        metrics_path = f"output/results/{cat}_{model}_reranking_metrics.json"
        try:
            with open(metrics_path, "r") as f:
                metrics_data = json.load(f)["per_lambda"]
            baseline_carbon = metrics_data[0]["avg_carbon_kg"]
            best = None
            for pt in sorted(metrics_data, key=lambda p: -p["NDCG@10"]):
                reduction = (baseline_carbon - pt["avg_carbon_kg"]) / max(baseline_carbon, 1e-9)
                if reduction >= 0.10:
                    best = pt
                    break
            if best:
                print(f"{cat:20s} | {model:8s} | λ={best['lambda']:.2f} | NDCG@10={best['NDCG@10']:.4f} | "
                      f"carbon={best['avg_carbon_kg']:.2f} kg (−{reduction*100:.1f}%)")
                best_points.append({"category": cat, "model": model, "lambda": best["lambda"], "NDCG@10": best["NDCG@10"], "avg_carbon_kg": best["avg_carbon_kg"]})
            else:
                print(f"{cat:20s} | {model:8s} | No point with ≥10% carbon reduction")
        except Exception as e:
            print(f"{cat:20s} | {model:8s} | Results missing: {e}")

print("\nResults, metrics, and plots are saved in the output/results/ and output/figures/ folders.")

# --- Plot: Best NDCG@10 vs. Carbon for Each Model (across categories) ---
import pandas as pd
if best_points:
    df = pd.DataFrame(best_points)
    plt.figure(figsize=(8, 6))
    for model in MODELS:
        sub = df[df["model"] == model]
        plt.scatter(sub["avg_carbon_kg"], sub["NDCG@10"], label=model, s=80)
        for _, row in sub.iterrows():
            plt.text(row["avg_carbon_kg"]+0.01, row["NDCG@10"]+0.002, row["category"], fontsize=9)
    plt.xlabel("Avg Carbon (kg CO₂e)")
    plt.ylabel("NDCG@10")
    plt.title("Best Pareto Points (≥10% carbon reduction)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# --- Plot: Best NDCG@10 vs. Carbon for Each Category (across models) ---
if best_points:
    plt.figure(figsize=(8, 6))
    for cat in CATEGORIES:
        sub = df[df["category"] == cat]
        plt.scatter(sub["avg_carbon_kg"], sub["NDCG@10"], label=cat, s=80)
        for _, row in sub.iterrows():
            plt.text(row["avg_carbon_kg"]+0.01, row["NDCG@10"]+0.002, row["model"], fontsize=9)
    plt.xlabel("Avg Carbon (kg CO₂e)")
    plt.ylabel("NDCG@10")
    plt.title("Best Pareto Points by Category (≥10% carbon reduction)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()